In [ ]:
import sys
import vega
import os
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import torch
import numpy as np
import torch.nn.functional as F
from torch import nn, optim
from scipy import sparse
import umap
import umap.plot
import ast

REPO_ROOT = Path("../../../../..").resolve()
VEGA_REPRO = REPO_ROOT / "models_reproductibility" / "vega-reproducibility"
sys.path.append(str(VEGA_REPRO / "src"))
import vanilla_vae
import train_vanilla_vae_suppFig1
import utils
from utils import *
from learning_utils import *
from vanilla_vae import VanillaVAE
from vega_model import VEGA
from customized_linear import CustomizedLinear

sys.path.append(str(REPO_ROOT / "utils" / "utils_evaluation"))
import utils_train_models
from utils_train_models import *
import vega_training
from vega_training import *
import vega_utils
from vega_utils import *
import utils_classification_emb
from utils_classification_emb import *


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


In [ ]:
pathway_file = str(VEGA_REPRO / "data" / "reactomes.gmt")
path_data = str(REPO_ROOT / "datasets" / "tischdb") + "/"


In [ ]:
name_dataset = "CHOL_GSE138709"
type_data = 'tisch'


In [ ]:
#Load data
if type_data == 'tisch':
    metadata = pd.read_csv(path_data + f'{name_dataset}_CellMetainfo_table.tsv', sep="\t")
    adata = sc.read_10x_h5(path_data + f'{name_dataset}_expression.h5', genome='GRCh38', gex_only=False)
    adata.obs = metadata
    column_labels_name = 'Celltype (major-lineage)' 
if name_dataset == 'Kang_PBMC':
    adata = sc.read(str(REPO_ROOT / "datasets" / "VEGA" / "Kang PBMC" / "train_pbmc.h5ad"))
    column_labels_name = 'cell_type' 

#prepare data
adata, pathway_dict, pathway_mask, list_pathways, df_genespathways, overlap_matrix = access_data_vega(None, adata, pathway_file)


In [ ]:
adata


# Preprocessing

In [ ]:
name_model = 'Vega'
select_hvg = True
n_top_genes = 2000
random_seed = 42
train_size = 0.9
preprocess = True


In [ ]:
adata, adata_train, adata_test, train_data, test_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, preprocess, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata, pathway_file)
adata, adata_train, adata_val, train_data, val_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, False, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata_train, pathway_file)
adata


In [ ]:
adata.X.toarray().max()


# Single training

In [ ]:
dir_path = str(REPO_ROOT / "notebooks" / "review VAEs" / "other datasets") + "/"
with open(dir_path+'parameters.txt', "r") as f:
    data_str = f.read() 
    dict_parameters = ast.literal_eval(data_str)
params = dict_parameters[name_model][name_dataset]
n_epochs = params['n_epochs']
lr = params['lr']
batch_size = params['batch_size']
beta = params['beta']
dropout = params['dropout']
val_resolution = params['val_resolution']
train_p = params['train_p']
test_p = params['test_p']
dev = params['dev']
save_path = params['save_path']
pos_dec = params['pos_dec']


In [ ]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True, drop_last=True)

dict_params = {
    "pathway_mask": pathway_mask,
    "n_pathways": pathway_mask.shape[1],
    "n_genes": pathway_mask.shape[0],
    "device": device,
    "beta": beta,
    "save_path": dir_path + '/vega/interpretability evaluation/',
    "dropout": dropout,
    "pos_dec": True,
}

model = VEGA(**dict_params).to(device)


hist, mse_hist = model.train_model(
    train_loader,
    lr,
    n_epochs,
    train_p,
    test_p,
    test_loader,
    save_model=True
)


In [ ]:
model


# Save

In [ ]:
path_to_save = f'/home/data/{name_model}/{name_dataset}/original'
path_to_save_reconstructed = path_to_save + '/reconstructed/'
path_to_save_embeddings = path_to_save + '/embeddings'

os.makedirs(path_to_save, exist_ok=True)
os.makedirs(path_to_save_reconstructed, exist_ok=True)
os.makedirs(path_to_save_embeddings, exist_ok=True)


In [ ]:
embeddings_train = model.to_latent(torch.Tensor(adata_train.X.toarray()).to(device))
X_reconstructed_train = model.decode(torch.Tensor(embeddings_train).to(device))

embeddings_val = model.to_latent(torch.Tensor(adata_val.X.toarray()).to(device))
X_reconstructed_val = model.decode(torch.Tensor(embeddings_val).to(device))

embeddings_test = model.to_latent(torch.Tensor(adata_test.X.toarray()).to(device))
X_reconstructed_test = model.decode(torch.Tensor(embeddings_test).to(device))


In [ ]:
from pathlib import Path
np.savetxt(Path(path_to_save_embeddings + f'/vega_{name_dataset}_embeddings_train_original.txt'), embeddings_train.cpu().detach().numpy())
np.savetxt(Path(path_to_save_reconstructed + f'/vega_{name_dataset}_reconstruction_train_original.txt'), X_reconstructed_train.cpu().detach().numpy())
np.savetxt(Path(path_to_save_embeddings + f'/vega_{name_dataset}_embeddings_val_original.txt'), embeddings_val.cpu().detach().numpy())
np.savetxt(Path(path_to_save_reconstructed + f'/vega_{name_dataset}_reconstruction_val_original.txt'), X_reconstructed_val.cpu().detach().numpy())
np.savetxt(Path(path_to_save_embeddings + f'/vega_{name_dataset}_embeddings_test_original.txt'), embeddings_test.cpu().detach().numpy())
np.savetxt(Path(path_to_save_reconstructed + f'/vega_{name_dataset}_reconstruction_test_original.txt'), X_reconstructed_test.cpu().detach().numpy())
